# Data Cleaning and Feature Engineering


Cleans and merges six datasets into a single training-ready CSV for the LSTM model. Produces `data/processed/cereals_merged_clean.csv` (1,744 rows) and `app/ml/models/price_scalers.pkl` (scalers needed for inference-time inverse transformation).

In [1]:
import os
project_root = os.path.abspath('..')
os.chdir(project_root)
print("Working directory:", os.getcwd())

Working directory: /Users/ac/Desktop/postharvest-iq


In [2]:
import pandas as pd
import numpy as np
import pickle
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import MinMaxScaler

load_dotenv()

db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

engine = create_engine(
    f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

print("connected to database successfully")

connected to database successfully


## 1. Load Market Prices
Wholesale cereal prices for Maize, Millet, and Sorghum across five markets — Tamale, Bolga, and Wa as the primary northern targets; Kumasi and Techiman as southern reference signals whose prices lead northern markets by 2–3 weeks.

In [3]:
query = """
    SELECT
        p.date,
        p.market,
        p.market_id,
        p.commodity,
        p.price,
        p.pricetype,
        m.latitude,
        m.longitude,
        m.admin1
    FROM wfp_prices p
    JOIN wfp_markets m ON p.market_id = m.market_id
    WHERE p.commodity IN ('Maize', 'Millet', 'Sorghum')
    AND p.pricetype = 'Wholesale'
    AND p.market IN ('Tamale', 'Bolga', 'Wa', 'Kumasi', 'Techiman')
    ORDER BY p.market, p.commodity, p.date
"""

df = pd.read_sql(query, engine)
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

print(f"total records loaded: {len(df)}")
print(f"markets: {sorted(df['market'].unique())}")
print(f"commodities: {sorted(df['commodity'].unique())}")
print(f"date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"missing values: {df.isnull().sum().sum()}")
print()
print("records per market per commodity:")
print(df.groupby(['market', 'commodity']).size().unstack(fill_value=0))

total records loaded: 1770
markets: ['Bolga', 'Kumasi', 'Tamale', 'Techiman', 'Wa']
commodities: ['Maize', 'Millet', 'Sorghum']
date range: 2006-01-15 to 2023-07-15
missing values: 0

records per market per commodity:
commodity  Maize  Millet  Sorghum
market                           
Bolga        108      78       71
Kumasi       206     188      127
Tamale       100     129       76
Techiman     163     195      118
Wa            67      74       70


## 2. Exchange Rate
Annual GHS/USD rate from FAOSTAT merged by year. Gives the LSTM a macro signal to separate inflation-driven price spikes (2022–2023 cedi collapse) from genuine seasonal supply-demand patterns.

In [4]:
fx_query = """
    SELECT year, value as exchange_rate
    FROM ghana_exchange_rates
    WHERE months = 'Annual value'
    AND element = 'Local currency units per USD'
    ORDER BY year
"""

fx = pd.read_sql(fx_query, engine)
fx['year'] = fx['year'].astype(int)

# 2006-2007 annual values are in old Ghana cedis — divide by 10,000 to convert
fx.loc[fx['exchange_rate'] >= 100, 'exchange_rate'] = (
    fx.loc[fx['exchange_rate'] >= 100, 'exchange_rate'] / 10_000
)

fx = fx.groupby('year')['exchange_rate'].mean().reset_index()

print(f"exchange rate records after cleaning: {len(fx)}")
print(f"year range: {fx['year'].min()} to {fx['year'].max()}")
print()
print("exchange rates from 2006 onwards:")
print(fx[fx['year'] >= 2006].to_string(index=False))

df = df.merge(fx, on='year', how='left')
df = df.drop_duplicates(subset=['date', 'market', 'commodity']).reset_index(drop=True)

print(f"\nrows after exchange rate merge: {len(df)}")
print(f"missing exchange rates: {df['exchange_rate'].isnull().sum()}")

exchange rate records after cleaning: 56
year range: 1970 to 2025

exchange rates from 2006 onwards:
 year  exchange_rate
 2006       0.915107
 2007       0.932619
 2008       1.052275
 2009       1.404967
 2010       1.429983
 2011       1.520625
 2012       1.824867
 2013       1.981350
 2014       2.896575
 2015       3.714642
 2016       3.909817
 2017       4.350533
 2018       4.585325
 2019       5.217367
 2020       5.595708
 2021       5.805700
 2022       8.272400
 2023      11.020409
 2024      14.181942
 2025      12.527120

rows after exchange rate merge: 1770
missing exchange rates: 0


## 3. Producer Price Index
Farm gate price index (2014–2016 = 100) from FAOSTAT merged by year and commodity. Captures the distress-selling spread: when farm gate prices sit far below market prices, distress selling is active and the subsequent recovery tends to be stronger.

In [5]:
# FAOSTAT has no maize producer prices for Ghana — millet/sorghum average used as proxy
pp_query = """
    SELECT year, item, value as producer_price_index
    FROM fao_producer_prices
    WHERE item IN ('Maize', 'Millet', 'Sorghum')
    AND months = 'Annual value'
    AND element = 'Producer Price Index (2014-2016 = 100)'
    ORDER BY item, year
"""

pp = pd.read_sql(pp_query, engine)
pp['year'] = pp['year'].astype(int)
pp = pp[pp['producer_price_index'] < 10000].copy()
pp = pp.groupby(['year', 'item'])['producer_price_index'].mean().reset_index()

pp_wide = pp.pivot(index='year', columns='item', values='producer_price_index').reset_index()
pp_wide.columns.name = None

# skipna=False requires both Millet and Sorghum to exist before computing the proxy
maize_proxy = pp_wide[['Millet', 'Sorghum']].mean(axis=1, skipna=False)
if 'Maize' in pp_wide.columns:
    pp_wide['Maize'] = pp_wide['Maize'].fillna(maize_proxy)
else:
    pp_wide['Maize'] = maize_proxy

print("producer price table from 2006 onwards:")
print(pp_wide[pp_wide['year'] >= 2006].to_string(index=False))

def get_producer_index(row):
    match = pp_wide[pp_wide['year'] == row['year']]
    if len(match) == 0:
        return None
    col = row['commodity']
    if col not in match.columns:
        return None
    return match[col].values[0]

df['producer_price_index'] = df.apply(get_producer_index, axis=1)

print(f"\nrows after producer price merge: {len(df)}")
print(f"missing producer index: {df['producer_price_index'].isnull().sum()}")
print()
print(df[['date', 'market', 'commodity', 'price', 'producer_price_index']].head(8))

producer price table from 2006 onwards:
 year      Millet   Sorghum       Maize
 2006  311.713333  239.8700  275.791667
 2007  286.790000  233.2600  260.025000
 2008  489.362500  408.6575  449.010000
 2009  552.125000  457.2650  504.695000
 2010  559.857500  464.7325  512.295000
 2011  608.145000  521.5650  564.855000
 2012   61.740000   65.2200   63.480000
 2013   76.320000   78.8100   77.565000
 2014   86.070000   88.9900   87.530000
 2015  102.770000  101.3000  102.035000
 2016 1113.040000  877.1775  995.108750
 2017  775.507500  569.8950  672.701250
 2018  999.647500  826.9850  913.316250
 2019  978.195000  757.3775  867.786250
 2020 1152.955000  979.8625 1066.408750
 2021 1837.755000 1483.3600 1660.557500
 2022 2633.450000 2106.7750 2370.112500
 2023  364.970000  351.4600  358.215000
 2024  384.780000  374.6000  379.690000
 2025  475.630000  455.1700  465.400000

rows after producer price merge: 1770
missing producer index: 0

        date market commodity  price  producer_price_i

## 4. Outlier Removal
IQR filter applied independently per market-commodity group to remove data entry errors while preserving genuine price extremes driven by the 2022–2023 currency shock.

In [6]:
# IQR multiplier of 5 preserves genuine 2022-2023 cedi-collapse prices that multiplier 3 removed
n_before_outlier = len(df)
print(f"records before outlier removal: {n_before_outlier}")

def remove_outliers(group):
    q1 = group['price'].quantile(0.25)
    q3 = group['price'].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - (5 * iqr)
    upper = q3 + (5 * iqr)
    return group[(group['price'] >= lower) & (group['price'] <= upper)]

df = df.groupby(
    ['market', 'commodity'], group_keys=False
).apply(remove_outliers).reset_index(drop=True)

print(f"records after outlier removal: {len(df)}")
print(f"records removed: {n_before_outlier - len(df)}")
print()
print("records per market per commodity after cleaning:")
print(df.groupby(['market', 'commodity']).size().unstack(fill_value=0))
print()
print(f"2023 records kept: {len(df[df['date'].dt.year == 2023])}")
print(f"date range: {df['date'].min().date()} to {df['date'].max().date()}")

records before outlier removal: 1770
records after outlier removal: 1744
records removed: 26

records per market per commodity after cleaning:
commodity  Maize  Millet  Sorghum
market                           
Bolga        106      78       71
Kumasi       199     188      127
Tamale        99     123       72
Techiman     159     193      118
Wa            67      74       70

2023 records kept: 44
date range: 2006-01-15 to 2023-07-15


/var/folders/68/cg8hvl455rngc_wr5vs3wmsh0000gn/T/ipykernel_7868/2019880233.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(remove_outliers).reset_index(drop=True)


## 5. Feature Engineering
Price scaled per market-commodity pair so the LSTM learns seasonal patterns, not price-level differences between markets. Exchange rate and producer price index scaled globally — they are national indicators, not market-specific. Month encoded as sin/cos so the model treats December and January as adjacent.

In [7]:
scalers = {}
df['price_scaled'] = 0.0

for (market, commodity), group in df.groupby(['market', 'commodity']):
    scaler = MinMaxScaler()
    idx = group.index
    df.loc[idx, 'price_scaled'] = scaler.fit_transform(df.loc[idx, ['price']]).flatten()
    scalers[f"{commodity}_{market}"] = scaler

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

print(f"scalers created: {len(scalers)}")
print(f"scaler keys: {list(scalers.keys())}")
print()
print(f"price_scaled range: {df['price_scaled'].min():.2f} to {df['price_scaled'].max():.2f}")
print(f"month_sin range: {df['month_sin'].min():.2f} to {df['month_sin'].max():.2f}")
print(f"month_cos range: {df['month_cos'].min():.2f} to {df['month_cos'].max():.2f}")
print()
print(df[['date', 'market', 'commodity', 'price', 'price_scaled', 'month_sin', 'month_cos']].head(6))

scalers created: 15
scaler keys: ['Maize_Bolga', 'Millet_Bolga', 'Sorghum_Bolga', 'Maize_Kumasi', 'Millet_Kumasi', 'Sorghum_Kumasi', 'Maize_Tamale', 'Millet_Tamale', 'Sorghum_Tamale', 'Maize_Techiman', 'Millet_Techiman', 'Sorghum_Techiman', 'Maize_Wa', 'Millet_Wa', 'Sorghum_Wa']

price_scaled range: 0.00 to 1.00
month_sin range: -1.00 to 1.00
month_cos range: -1.00 to 1.00

        date market commodity  price  price_scaled  month_sin     month_cos
0 2006-01-15  Bolga     Maize  22.00      0.019197   0.500000  8.660254e-01
1 2006-09-15  Bolga     Maize  16.88      0.004890  -1.000000 -1.836970e-16
2 2006-10-15  Bolga     Maize  16.00      0.002431  -0.866025  5.000000e-01
3 2006-11-15  Bolga     Maize  15.13      0.000000  -0.500000  8.660254e-01
4 2007-01-15  Bolga     Maize  17.33      0.006148   0.500000  8.660254e-01
5 2007-03-15  Bolga     Maize  20.50      0.015006   1.000000  6.123234e-17


In [8]:
fx_scaler = MinMaxScaler()
df['fx_scaled'] = fx_scaler.fit_transform(df[['exchange_rate']]).flatten()
scalers['exchange_rate'] = fx_scaler

pp_scaler = MinMaxScaler()
df['pp_scaled'] = pp_scaler.fit_transform(df[['producer_price_index']]).flatten()
scalers['producer_price_index'] = pp_scaler

print(f"total scalers: {len(scalers)}")
print(f"fx_scaled range: {df['fx_scaled'].min():.2f} to {df['fx_scaled'].max():.2f}")
print(f"pp_scaled range: {df['pp_scaled'].min():.2f} to {df['pp_scaled'].max():.2f}")
print()
print(df[['date', 'market', 'commodity', 'price_scaled', 'month_sin', 'month_cos', 'fx_scaled', 'pp_scaled']].head(6))

total scalers: 17
fx_scaled range: 0.00 to 1.00
spread_scaled range: 0.00 to 1.00

        date market commodity  price_scaled  month_sin     month_cos  \
0 2006-01-15  Bolga     Maize      0.019197   0.500000  8.660254e-01   
1 2006-09-15  Bolga     Maize      0.004890  -1.000000 -1.836970e-16   
2 2006-10-15  Bolga     Maize      0.002431  -0.866025  5.000000e-01   
3 2006-11-15  Bolga     Maize      0.000000  -0.500000  8.660254e-01   
4 2007-01-15  Bolga     Maize      0.006148   0.500000  8.660254e-01   
5 2007-03-15  Bolga     Maize      0.015006   1.000000  6.123234e-17   

   fx_scaled  spread_scaled  
0   0.000000       0.083233  
1   0.000000       0.083233  
2   0.000000       0.083233  
3   0.000000       0.083233  
4   0.001733       0.077102  
5   0.001733       0.077102  


## 6. Save Outputs
Scalers must be saved alongside the model — they are required at inference time to convert normalised LSTM output back to Ghana cedis before presenting to the farmer.

In [9]:
import os

os.makedirs('app/ml/models', exist_ok=True)

with open('app/ml/models/price_scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)

print(f"saved {len(scalers)} scalers to app/ml/models/price_scalers.pkl")

final_cols = [
    'date', 'market', 'commodity', 'price',
    'latitude', 'longitude',
    'price_scaled', 'month_sin', 'month_cos',
    'fx_scaled', 'pp_scaled'
]

df_clean = df[final_cols].copy()

n_before_drop = len(df_clean)
df_clean = df_clean.dropna().reset_index(drop=True)
print(f"rows dropped by dropna (missing fx or producer index): {n_before_drop - len(df_clean)}")

os.makedirs('data/processed', exist_ok=True)
df_clean.to_csv('data/processed/cereals_merged_clean.csv', index=False)

print()
print(f"total rows: {len(df_clean)}")
print(f"columns: {list(df_clean.columns)}")
print(f"markets: {sorted(df_clean['market'].unique())}")
print(f"commodities: {sorted(df_clean['commodity'].unique())}")
print(f"date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
print(f"missing values: {df_clean.isnull().sum().sum()}")
print()
print("records per market per commodity:")
print(df_clean.groupby(['market', 'commodity']).size().unstack(fill_value=0))
print()
print(f"scalers file exists: {os.path.exists('app/ml/models/price_scalers.pkl')}")
print(f"clean csv exists: {os.path.exists('data/processed/cereals_merged_clean.csv')}")

saved 17 scalers to app/ml/models/price_scalers.pkl

total rows: 1744
columns: ['date', 'market', 'commodity', 'price', 'latitude', 'longitude', 'price_scaled', 'month_sin', 'month_cos', 'fx_scaled', 'spread_scaled']
markets: ['Bolga', 'Kumasi', 'Tamale', 'Techiman', 'Wa']
commodities: ['Maize', 'Millet', 'Sorghum']
date range: 2006-01-15 00:00:00 to 2023-07-15 00:00:00
missing values: 0

records per market per commodity:
commodity  Maize  Millet  Sorghum
market                           
Bolga        106      78       71
Kumasi       199     188      127
Tamale        99     123       72
Techiman     159     193      118
Wa            67      74       70

scalers file exists: True
clean csv exists: True
